# Food Access Prioritization Pipeline
Version 1

**Public Service & Food Access**

This notebook loads the USDA 2019 Large Retailer Access Map (LRAM), filters to a chosen county, cleans it, creates a prioritization score, and exports analysis-ready outputs.

**Design goal:** the county is a single config variable, so the whole analysis can be re-run for any other county without editing the logic.

> **Data note:** LRAM is historical — 2010 census-tract boundaries & population, 2014–18 ACS socioeconomics, and 2019 retailer locations. It is *not* a 2026 measure of food access.

# Step 1: Load the USDA Data

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# To analyze a different county, change the state and county below
STATE = "Illinois"
COUNTY = "Cook County"

# Data file must be in the same folder as the notebook
DATA_FILE = "FoodAccessResearchAtlasData2019.xlsx"
SHEET_NAME = "Food Access Research Atlas"

In [3]:
# Load the USDA data
df = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)

# Check the size of the original dataset
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

Rows: 72,531
Columns: 147


In [4]:
# Display the first five rows
df.head()

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,1001020100,Alabama,Autauga County,1,1912,693,0,0.0,0.000000,0,...,221.0,1622.0,217.0,14.0,0.0,14.0,45.0,44.0,6.0,102.0
1,1001020200,Alabama,Autauga County,1,2170,743,0,181.0,8.341014,1,...,214.0,888.0,1217.0,5.0,0.0,5.0,55.0,75.0,89.0,156.0
2,1001020300,Alabama,Autauga County,1,3373,1256,0,0.0,0.000000,0,...,439.0,2576.0,647.0,17.0,5.0,11.0,117.0,87.0,99.0,172.0
3,1001020400,Alabama,Autauga County,1,4386,1722,0,0.0,0.000000,0,...,904.0,4086.0,193.0,18.0,4.0,11.0,74.0,85.0,21.0,98.0
4,1001020500,Alabama,Autauga County,1,10766,4082,0,181.0,1.681219,0,...,1126.0,8666.0,1437.0,296.0,9.0,48.0,310.0,355.0,230.0,339.0


# Step 2: Define the Variables for Analysis

In [20]:
# Variables needed for the analysis
analysis_variables = [
    "CensusTract", 
    "LILATracts_1And10", 
    "PovertyRate", 
    "MedianFamilyIncome", 
    "lapop1share", 
    "lalowi1share", 
    "lahunv1share", 
    "lasnap1share",
    "Pop2010"]

# Step 3: Filter the Data by State and County

In [21]:
# Create new dataset
county_data = df[(df["State"] == STATE) & (df["County"] == COUNTY)].copy()

In [7]:
# Error if an invalid state or county is entered 
if county_data.empty: 
    raise ValueError( 
        f"No records found for {COUNTY} County, {STATE}. " 
        "Check the state and county names." 
    )

In [22]:
# Display the number of records selected 
print(f"Selected area: {COUNTY} County, {STATE}") 
print(f"Census tracts: {len(county_data):,}")

Selected area: Cook County County, Illinois
Census tracts: 1,314


# Step 4: Create the Analysis Dataset

In [23]:
# Keep only the variables needed for the analysis 
analysis_data = county_data[analysis_variables].copy()

# Display the size of the analysis dataset 
print(f"Analysis rows: {len(analysis_data):,}") 
print(f"Analysis columns: {len(analysis_data.columns):,}")

Analysis rows: 1,314
Analysis columns: 9


In [24]:
# Display the first five rows 
analysis_data.head()

,CensusTract,LILATracts_1And10,PovertyRate,MedianFamilyIncome,lapop1share,lalowi1share,lahunv1share,lasnap1share,Pop2010
20935,17031010100,0,35.796871,62177.0,NaN,NaN,NaN,NaN,4854
20936,17031010201,0,36.839858,47411.0,NaN,NaN,NaN,NaN,6450
20937,17031010202,0,18.458498,51719.0,NaN,NaN,NaN,NaN,2818
20938,17031010300,0,16.149506,66875.0,NaN,NaN,NaN,NaN,6236
20939,17031010400,0,19.060698,59861.0,NaN,NaN,NaN,NaN,5042
20940,17031010501,0,20.700483,50266.0,NaN,NaN,NaN,NaN,4091
20941,17031010502,0,30.032644,37500.0,NaN,NaN,NaN,NaN,3101
20942,17031010503,0,48.044367,43654.0,NaN,NaN,NaN,NaN,2290
20943,17031010600,0,24.389044,87545.0,NaN,NaN,NaN,NaN,6097
20944,17031010701,0,20.791290,72500.0,NaN,NaN,NaN,NaN,3864


# Step 5: Check Data Quality

In [18]:
# Check for missing values 
missing_values = analysis_data.isna().sum()

print("Missing values by variable:") 
print(missing_values)

Missing values by variable:
LILATracts_1And10       0
PovertyRate             0
MedianFamilyIncome      9
lapop1share           972
lalowi1share          972
lahunv1share          972
lasnap1share          972
Pop2010                 0
dtype: int64


In [19]:
# Check for duplicate census tracts 
duplicate_tracts = analysis_data["CensusTract"].duplicated().sum() 

print(f"Duplicate census tracts: {duplicate_tracts}")

KeyError: 'CensusTract'

In [ ]:
# Check the data types
analysis_data.dtypes

In [ ]:
# Generate summary statistics for numeric variables 
analysis_data.describe().T